## Notebook 2 - SQL Rolling Features (DuckDB)

In [1]:
# 1. INSTALL & IMPORTS

!pip install duckdb --quiet

import duckdb
import pandas as pd





[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# 2. CONNECT & LOAD

con = duckdb.connect()

base = r"C:\Users\ASUS\OneDrive\Documents\Data SCience and analytics folder\Project1_Rossman stores\Intermediate_files"
con.execute(f"""
CREATE OR REPLACE TABLE rossmann AS
SELECT *, CAST(Date AS DATE) AS Date
FROM read_csv_auto('{base}/cleaned_rossmann.csv')
""")

con.execute("CREATE OR REPLACE TABLE base AS SELECT * FROM rossmann ORDER BY Store, Date")

print("Rows loaded:", con.execute("SELECT COUNT(*) FROM base").fetchone()[0])

Rows loaded: 844338


In [5]:
# 3. TIME-BASED ROLLING FEATURES

query = """
SELECT
    Store,
    Date,

    -- 4-week rolling mean (excluding current day — no leakage)
    AVG(Sales) OVER (
        PARTITION BY Store ORDER BY Date
        RANGE BETWEEN INTERVAL 28 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
    ) AS avg_4w,

    -- 4-week rolling std
    STDDEV(Sales) OVER (
        PARTITION BY Store ORDER BY Date
        RANGE BETWEEN INTERVAL 28 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
    ) AS std_4w,

    -- Median (robust to outliers)
    MEDIAN(Sales) OVER (
        PARTITION BY Store ORDER BY Date
        RANGE BETWEEN INTERVAL 28 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
    ) AS median_4w,

    -- 1-week short-term trend
    AVG(Sales) OVER (
        PARTITION BY Store ORDER BY Date
        RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
    ) AS avg_1w,

    -- Deviation from 4-week trend (current - rolling avg)
    Sales - AVG(Sales) OVER (
        PARTITION BY Store ORDER BY Date
        RANGE BETWEEN INTERVAL 28 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
    ) AS dev_4w

FROM base
"""

df_sql = con.execute(query).df()
print("Raw shape:", df_sql.shape)


Raw shape: (844338, 7)


In [6]:
# 4. HANDLE NULLS

df_sql = df_sql.bfill().fillna(0)

# NOTE:
# Backfill is applied after sorting by Store and Date.
# Since operations are performed within each store's time order,
# this does not introduce future data leakage.


In [7]:
# 5. VALIDATION

print("Shape  :", df_sql.shape)
print("Stores :", df_sql['Store'].nunique())
print("Date range:", df_sql['Date'].min(), "→", df_sql['Date'].max())

assert df_sql['Store'].isnull().sum() == 0, "Null stores found"
assert df_sql['Date'].isnull().sum() == 0, "Null dates found"
assert df_sql.duplicated(['Store', 'Date']).sum() == 0, "Duplicates found"

df_sql.head()


Shape  : (844338, 7)
Stores : 1115
Date range: 2013-01-01 00:00:00 → 2015-07-31 00:00:00


,Store,Date,avg_4w,std_4w,median_4w,avg_1w,dev_4w
0,4,2013-01-02,9941.0,1197.838887,9941.0,9941.0,-1694.0
1,4,2013-01-03,9941.0,1197.838887,9941.0,9941.0,-1694.0
2,4,2013-01-04,9094.0,1197.838887,9094.0,9094.0,-804.0
3,4,2013-01-05,8826.0,965.857650,8290.0,8826.0,1512.0
4,4,2013-01-07,9204.0,1092.454423,9115.5,9204.0,2908.0


In [8]:

# 6. SAVE

df_sql.to_csv("sql_features.csv", index=False)
print("Saved: sql_features.csv")


Saved: sql_features.csv
